# MNIST 데이터셋으로 신경망 추론 (배치 처리)

이 노트북에서는 배치(batch) 단위로 MNIST 손글씨 숫자 이미지 인식 신경망을 구동합니다.

## 배치 처리란?

배치 처리는 여러 데이터를 한 번에 묶어서 처리하는 방식입니다.
하나씩 처리하는 것보다 GPU/컴퓨터의 병렬 연산 능력을 효율적으로 활용할 수 있습니다.

## 신경망 구조

| 층 | 뉴런 수 | 활성화 함수 |
|----|---------|------------|
| 입력층 | 784 | - |
| 은닉층 1 | 50 | sigmoid |
| 은닉층 2 | 100 | sigmoid |
| 출력층 | 10 | softmax |

---

## 📌 실습 과제: MNIST 신경망 추론 (배치 처리)

이 섹션의 코드는 이후 실습 과제에서 `import` 형태로 재사용됩니다.

In [ ]:
# coding: utf-8
# ============================================
# 실습 과제: MNIST 신경망 추론 (배치 처리)
# ============================================

# 필요한 라이브러리 Import
import sys, os
from pathlib import Path

# Library import를 위한 현재 디렉토리 변경
print(os.getcwd())
current_dir = os.path.dirname(os.getcwd())
print(current_dir)
os.chdir(current_dir)

In [ ]:
# ============================================
# 실습 과제: 추가 Library import
# ============================================

import numpy as np
import pickle
from dataset.mnist import load_mnist
from common.functions import sigmoid, softmax

In [ ]:
# ============================================
# 실습 과제: 데이터 로드 함수
# ============================================

def get_data():
    """
    MNIST 데이터를 로드하고 반환합니다.
    
    Returns
    -------
    tuple: (테스트 이미지, 테스트 레이블)
    """
    # 숫자 데이터 다운로드 from internet
    # normalize=True: 픽셀 값을 0.0~1.0으로 정규화
    # flatten=True: 이미지를 1차원 배열로 펼침 (784차원)
    # one_hot_label=False: 레이블을 정수 인덱스로 반환
    (x_train, t_train), (x_test, t_test) = load_mnist(
        normalize=True, flatten=True, one_hot_label=False
    )
    return x_test, t_test

In [ ]:
# ============================================
# 실습 과제: 신경망 초기화 함수
# ============================================

def init_network():
    """
    사전 학습된 가중치 파일을 로드하여 신경망을 초기화합니다.
    
    Returns
    -------
    dict: 가중치(W)와 편향(b)을 포함하는 신경망 파라미터
    """
    # 신경망 가중치 값 로딩
    with open("ch02/sample_weight.pkl", "rb") as f:
        network = pickle.load(f)
    
    return network

In [ ]:
# ============================================
# 실습 과제: 예측 (순전파) 함수
# ============================================

def predict(network, x):
    """
    신경망을 통해 입력 데이터 x의 출력을 예측합니다.
    배치(batch) 단위로 여러 이미지를 한 번에 처리할 수 있습니다.
    
    Parameters
    ----------
    network : dict
        초기화된 신경망 (가중치와 편향 포함)
    x : numpy array
        입력 데이터 (N, 784) 형태 - N개의 이미지
    
    Returns
    -------
    numpy array: 출력층의 확률 분포 (N, 10) 형태
    """
    # 가중치와 편향 추출
    W1, W2, W3 = network["W1"], network["W2"], network["W3"]
    b1, b2, b3 = network["b1"], network["b2"], network["b3"]

    # 신경망 출력값 계산
    # 입력층 → 은닉층 1
    a1 = np.dot(x, W1) + b1
    z1 = sigmoid(a1)
    
    # 은닉층 1 → 은닉층 2
    a2 = np.dot(z1, W2) + b2
    z2 = sigmoid(a2)
    
    # 은닉층 2 → 출력층
    a3 = np.dot(z2, W3) + b3
    y = softmax(a3)

    return y

In [ ]:
# ============================================
# 실습 과제: 배치 단위 정확도 계산
# ============================================

# 데이터 로드
x, t = get_data()

# 신경망 초기화 (가중치 로드)
network = init_network()

# 배치(묶음)의 크기 지정
batch_size = 100

# 정확도 카운터 초기화
accuracy_cnt = 0

# 배치 단위 for 루프
# range(시작, 끝, 단계) - 0부터 시작하여 batch_size만큼 증가
for i in range(0, len(x), batch_size):
    # 현재 배치의 데이터 추출
    x_batch = x[i:i+batch_size]
    
    # 배치(묶음) 단위로 예측 작업 수행
    y_batch = predict(network, x_batch)
    
    # 각 샘플의 확률이 가장 높은 원소의 인덱스를 얻는다 (axis=1: 행 방향)
    p = np.argmax(y_batch, axis=1)
    
    # 맞춘 개수 누적
    accuracy_cnt += np.sum(p == t[i:i+batch_size])

# 최종 정확도 출력
print(f"Accuracy: {str(float(accuracy_cnt) / len(x))}")

---

## 📖 설명: 배치 처리 이해하기

이 섹션에서는 배치 처리가 어떻게 동작하는지 자세히 설명합니다.

### 전체 흐름 요약

```
1. 데이터 로드 (get_data)
   ↓
2. 신경망 초기화 (init_network) - 가중치 파일 로드
   ↓
3. 배치 단위 예측 (predict) - batch_size만큼 묶어서 처리
   ↓
4. 배치 단위 정확도 계산 - np.sum()으로 맞춘 개수 누적
```

In [ ]:
# 데이터 로드 및 신경망 초기화
x, t = get_data()
network = init_network()

# 로드된 데이터와 신경망 정보 확인
print(f"테스트 이미지 개수: {len(x)}")
print(f"테스트 이미지 형태: {x[0].shape}")
print(f"테스트 레이블 형태: {t.shape}")

print("\n신경망 파라미터 정보:")
for key, value in network.items():
    print(f"  {key}: 형태 = {value.shape}")

print(f"\n배치 크기: {batch_size}")
print(f"처리할 배치 수: {(len(x) + batch_size - 1) // batch_size}")

In [ ]:
# 배치 처리의 개념 시각화
import matplotlib.pyplot as plt

# 배치 처리 과정 설명
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

# 전체 데이터 10,000개 중 첫 3개 배치만 시각화
num_batchesToShow = min(3, (len(x) + batch_size - 1) // batch_size)

for batch_idx in range(num_batchesToShow):
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, len(x))
    
    # 배치 데이터
    x_batch = x[start_idx:end_idx]
    t_batch = t[start_idx:end_idx]
    
    # 예측 수행
    y_batch = predict(network, x_batch)
    p_batch = np.argmax(y_batch, axis=1)
    
    # 배치 내 첫 번째 이미지 표시
    img_idx = 0
    img = x_batch[img_idx].reshape(28, 28)
    
    ax.imshow(img, cmap='gray')
    ax.set_title(f"배치 {batch_idx + 1}: 인덱스 {start_idx}~{end_idx-1}\n"
                 f"이미지: {t_batch[img_idx]}, 예측: {p_batch[img_idx]}")
    ax.axis('off')

plt.suptitle("배치 처리 개념도", fontsize=14)
plt.tight_layout()
plt.show()

### 배치 처리 vs 단일 처리 비교

| 방식 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **단일 처리** | 한 번에 1개 이미지 처리 | 메모리 사용량 적음 | 처리 속도 느림 |
| **배치 처리** | 한 번에 N개 이미지 처리 | GPU 병렬 연산 활용, 속도 빠름 | 메모리 사용량 많음 |

**배치 처리 코드 비교:**

```python
# 단일 처리 (neuralnet_mnist.ipynb)
for i in range(len(x)):
    y = predict(network, x[i])      # 1개 처리
    p = np.argmax(y)                # 스칼라 결과
    if p == t[i]:
        accuracy_cnt += 1

# 배치 처리 (neuralnet_mnist_batch.ipynb)
for i in range(0, len(x), batch_size):
    x_batch = x[i:i+batch_size]     # batch_size개 처리
    y_batch = predict(network, x_batch)
    p = np.argmax(y_batch, axis=1)  # 배열 결과
    accuracy_cnt += np.sum(p == t[i:i+batch_size])
```

In [ ]:
# 배치 처리 과정 단계별 확인
batch_size_demo = 5  # 데모용 작은 배치 크기

x_demo = x[:batch_size_demo]
t_demo = t[:batch_size_demo]

print(f"데모 배치 크기: {batch_size_demo}")
print(f"입력 형태: {x_demo.shape}")  # (5, 784)

# 신경망 출력값 계산
W1, W2, W3 = network["W1"], network["W2"], network["W3"]
b1, b2, b3 = network["b1"], network["b2"], network["b3"]

# 은닉층 1
a1 = np.dot(x_demo, W1) + b1
z1 = sigmoid(a1)
print(f"\n은닉층 1:")
print(f"  a1 형태: {a1.shape}")  # (5, 50)
print(f"  z1 형태: {z1.shape}")  # (5, 50)

# 은닉층 2
a2 = np.dot(z1, W2) + b2
z2 = sigmoid(a2)
print(f"\n은닉층 2:")
print(f"  a2 형태: {a2.shape}")  # (5, 100)
print(f"  z2 형태: {z2.shape}")  # (5, 100)

# 출력층
a3 = np.dot(z2, W3) + b3
y = softmax(a3)
print(f"\n출력층:")
print(f"  a3 형태: {a3.shape}")  # (5, 10)
print(f"  y 형태: {y.shape}")    # (5, 10)

# 예측 결과
p = np.argmax(y, axis=1)
print(f"\n예측 클래스: {p}")
print(f"정답 클래스: {t_demo}")
print(f"정답 개수: {np.sum(p == t_demo)}/{batch_size_demo}")

### 배치 크기 (Batch Size)의 영향

배치 크기를 변경하면 처리 속도와 정확도에 어떤 영향이 있는지 확인합니다.

In [ ]:
# 다양한 배치 크기로 정확도 확인
x_test, t_test = get_data()
network_test = init_network()

batch_sizes = [1, 10, 50, 100, 500, 1000]

print(f"{'배치 크기':<12} {'정확도':<12} {'처리된 개수':<15}")
print("-" * 40)

for bs in batch_sizes:
    accuracy_cnt = 0
    total_processed = 0
    
    for i in range(0, len(x_test), bs):
        x_batch = x_test[i:i+bs]
        y_batch = predict(network_test, x_batch)
        p = np.argmax(y_batch, axis=1)
        accuracy_cnt += np.sum(p == t_test[i:i+bs])
        total_processed += len(x_batch)
    
    accuracy = float(accuracy_cnt) / total_processed
    print(f"{bs:<12} {accuracy:<12.4f} {total_processed:<15}")

---

## 보충 자료: 배치 처리 이해하기

### np.argmax()의 axis 파라미터

배치 처리에서는 `axis=1`을 사용하여 행 방향(각 샘플별)으로 최대값 인덱스를 찾습니다.

```python
# axis=0: 열 방향 (배치 간 비교)
# axis=1: 행 방향 (각 샘플별 예측)

y_batch = np.array([[0.1, 0.7, 0.2],  # 샘플 0
                    [0.3, 0.2, 0.5]]) # 샘플 1

np.argmax(y_batch, axis=1)  # [1, 2] - 각 샘플의 최대값 인덱스
np.argmax(y_batch, axis=0)  # [1, 0, 1] - 각 클래스의 최대값 샘플 인덱스
```

In [ ]:
# np.argmax() axis 파라미터 예시
y_example = np.array([[0.1, 0.7, 0.2],
                      [0.3, 0.2, 0.5],
                      [0.8, 0.1, 0.1]])

print(f"출력 확률 행렬:\n{y_example}")
print(f"\naxis=0 (열 방향 최대값 인덱스): {np.argmax(y_example, axis=0)}")
print(f"axis=1 (행 방향 최대값 인덱스): {np.argmax(y_example, axis=1)}")

### 주요 개념 요약

| 개념 | 설명 |
|------|------|
| **배치 (Batch)** | 여러 데이터를 한 번에 묶어서 처리하는 단위 |
| **배치 크기 (Batch Size)** | 한 번에 처리할 데이터 개수 |
| **np.argmax(y, axis=1)** | 각 샘플의 최대 확률 인덱스 (예측 클래스) |
| **np.sum(p == t)** | 맞춘 예측 개수 계산 |
| **range(0, len(x), batch_size)** | 배치 단위로 반복하는 range |
| **x[i:i+batch_size]** | 현재 배치의 데이터 슬라이싱 |